# GTF Exon skipping finder

In [20]:
import re

filename = "chr10.gtf"

def read_gtf_exons(filename):
    with open(filename) as f:
        for line in f:
            if line.split('\t')[2] == 'exon':
                yield line.strip()

def get_exon(exon_record):
    fields = exon_record.split('\t')
    transcript_id = re.search(r'transcript_id\s+"(.+?)"', fields[8]).group(1)
    start = int(fields[3])
    end = int(fields[4])
    strand = fields[6]
    return (transcript_id, start, end, strand)

# Esoni per trascritto

In [21]:
exons_per_transcript = {}

for exon_record in read_gtf_exons(filename):
    transcript_id, start, end, strand =  get_exon(exon_record)
    transcript_dict = exons_per_transcript.setdefault(transcript_id.split('.')[0], {'strand': strand, 'exons': []})
    transcript_dict['exons'].append((start, end))

In [22]:
exons_per_transcript

{'ENST00000729394': {'strand': '+',
  'exons': [(10781, 11242), (12261, 12409), (12798, 13109)]},
 'ENST00000729392': {'strand': '+',
  'exons': [(10784, 11242), (12798, 13000), (15263, 15997)]},
 'ENST00000729389': {'strand': '+',
  'exons': [(10848, 11242),
   (12798, 13000),
   (15263, 15403),
   (42753, 42846),
   (43266, 43395),
   (43628, 43869)]},
 'ENST00000729390': {'strand': '+',
  'exons': [(10849, 11242),
   (12798, 13000),
   (15263, 15403),
   (42753, 42846),
   (43211, 43395),
   (43628, 43867)]},
 'ENST00000729391': {'strand': '+',
  'exons': [(10854, 11242),
   (12798, 13000),
   (42748, 42846),
   (43266, 43395),
   (43628, 43867)]},
 'ENST00000729393': {'strand': '+',
  'exons': [(11458, 12029), (12798, 13000), (15263, 15993)]},
 'ENST00000566940': {'strand': '-',
  'exons': [(13839, 14299), (16502, 16544), (20786, 20954)]},
 'ENST00000729478': {'strand': '-', 'exons': [(13851, 14299), (20786, 20931)]},
 'ENST00000729479': {'strand': '-', 'exons': [(13877, 16544), (2

# Dizionario gene-trascritti

In [23]:
filename = "gene-transcript-map.txt"

gene_transcripts_dict = {}

with open(filename) as f:
    next(f, None)
    for line in f:
        gene_id, transcript_id = line.strip().split()
        if transcript_id in exons_per_transcript:
            gene_transcripts_dict.setdefault(gene_id, {'ref': None, 'transcripts': set()})['transcripts'].add(transcript_id)

# Trascritto di riferimento

In [24]:
for gene, gene_dict in gene_transcripts_dict.items():
    transcripts = gene_dict['transcripts']
    if transcripts:
        gene_dict['ref'] = max(transcripts, key=lambda t: len(exons_per_transcript[t]['exons']))

In [25]:
gene_transcripts_dict

{'ENSG00000231743': {'ref': 'ENST00000419388',
  'transcripts': {'ENST00000419388'}},
 'ENSG00000226694': {'ref': 'ENST00000435539',
  'transcripts': {'ENST00000435539'}},
 'ENSG00000287653': {'ref': 'ENST00000653889',
  'transcripts': {'ENST00000653889'}},
 'ENSG00000237500': {'ref': 'ENST00000431209',
  'transcripts': {'ENST00000431209'}},
 'ENSG00000233052': {'ref': 'ENST00000432987',
  'transcripts': {'ENST00000432987'}},
 'ENSG00000227912': {'ref': 'ENST00000658923',
  'transcripts': {'ENST00000421077', 'ENST00000658923'}},
 'ENSG00000271251': {'ref': 'ENST00000603398',
  'transcripts': {'ENST00000603398'}},
 'ENSG00000185736': {'ref': 'ENST00000381312',
  'transcripts': {'ENST00000381305',
   'ENST00000381310',
   'ENST00000381312',
   'ENST00000469464',
   'ENST00000474762',
   'ENST00000477140',
   'ENST00000490172'}},
 'ENSG00000236420': {'ref': 'ENST00000425516',
  'transcripts': {'ENST00000425516'}},
 'ENSG00000235838': {'ref': 'ENST00000450068',
  'transcripts': {'ENST00000

# Exon skipping

In [26]:
def exon_skipping_finder(ref, tr, strand, id_ref, id_tr):
    sorted_tr = sorted(tr, key=lambda x: x[0], reverse=(strand == '-'))
    d, a, s = (1, 0, 1) if strand == '+' else (0, 1, -1)
    events = []
    sr = 0
    for exon1_tr, exon2_tr in zip(sorted_tr, sorted_tr[1:]):
        head = None
        for i, exon_ref in enumerate(ref[sr:], start=sr):
            if head is None:
                if exon_ref[d] == exon1_tr[d]:
                    head = i
                elif s * exon_ref[d] > s * exon1_tr[d]:
                    sr = i
                    break
            elif exon_ref[a] == exon2_tr[a]:
                count = i - head - 1
                if count > 0:
                    event_type = "ME" if count > 1 else "CE"
                    record = (id_tr, id_ref, event_type, count) if count > 1 else (id_tr, id_ref, event_type)
                    events.append(record)
                sr = i
                break
            elif s * exon_ref[a] > s * exon2_tr[a]:
                sr = i - 1
                break 
        else:
            return events
    return events

In [27]:
for gene, transcripts in gene_transcripts_dict.items():
    reference = transcripts['ref']
    sorted_ref_exons = sorted(exons_per_transcript[reference]['exons'],
                              key=lambda x: x[0],
                              reverse=(exons_per_transcript[reference]['strand'] == '-'))
    for transcript in transcripts['transcripts']:
        if transcript == reference:
            continue;
        events = exon_skipping_finder(sorted_ref_exons,
                                      exons_per_transcript[transcript]['exons'],
                                      exons_per_transcript[reference]['strand'],
                                      reference,
                                      transcript)
        if events:
            print(*events, sep='\n')   

('ENST00000484642', 'ENST00000298032', 'ME', 2)
('ENST00000468003', 'ENST00000298032', 'CE')
('ENST00000496741', 'ENST00000298032', 'CE')
('ENST00000496741', 'ENST00000298032', 'ME', 2)
('ENST00000376528', 'ENST00000298032', 'CE')
('ENST00000376528', 'ENST00000298032', 'ME', 2)
('ENST00000447081', 'ENST00000298032', 'CE')
('ENST00000377266', 'ENST00000454679', 'ME', 2)
('ENST00000377602', 'ENST00000648997', 'CE')
('ENST00000483339', 'ENST00000496261', 'CE')
('ENST00000373880', 'ENST00000486074', 'CE')
('ENST00000659541', 'ENST00000652859', 'CE')
('ENST00000371994', 'ENST00000536010', 'ME', 2)
('ENST00000464741', 'ENST00000327946', 'CE')
('ENST00000415731', 'ENST00000628416', 'ME', 2)
('ENST00000443748', 'ENST00000671701', 'CE')
('ENST00000443748', 'ENST00000671701', 'ME', 3)
('ENST00000434189', 'ENST00000666640', 'CE')
('ENST00000669629', 'ENST00000666640', 'CE')
('ENST00000669629', 'ENST00000666640', 'CE')
('ENST00000660354', 'ENST00000666640', 'CE')
('ENST00000666262', 'ENST000006666